## Transformación de datos
Pipeline mínimo: carga, filtrado, clasificación, costo y exportación.

In [ ]:
import pandas as pd
from google import genai
import numpy as np
from dotenv import load_dotenv
import os
import re
import ast

import random

In [ ]:
data = pd.read_csv('../data/raw/asistencias.csv')

In [ ]:
load_dotenv()
gemini_key = os.environ.get('OPENAI_API_KEY')
client = genai.Client(api_key=gemini_key)

In [ ]:
data

### Etapa 1: Filtrado de casos objetivo
Nos quedamos con asistencias canceladas y luego coordinadas.

In [ ]:
recoord_with_a_cost = data.query('cancelado_activado == 1 & coordinado == 1').copy()

In [ ]:
recoord_with_a_cost

In [ ]:
comments = "\n".join(recoord_with_a_cost['comentario'].to_list())

In [ ]:
comments

### Etapa 2: Clasificación de motivo
Convertimos comentarios en categorías cortas usando el modelo.

In [ ]:
resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"Analiza esta lista de python y devuelve una de la misma longitud con la razon de cada recoordinacion en menos de 4 palabras: \n {comments}"
)

In [ ]:
text = resp.candidates[0].content.parts[0].text

In [ ]:
match = re.search(r"```python\s*(.*?)\s*```", text, re.DOTALL)

if match:
    code = match.group(1)
    reason_list = ast.literal_eval(code)

print(reason_list)

In [ ]:
len(recoord_with_a_cost['comentario']) == len(reason_list), len(reason_list)

In [ ]:
recoord_with_a_cost['type'] = reason_list

In [ ]:
recoord_with_a_cost['type'].unique()

### Etapa 3: Costo operativo
Asignamos un costo por caso para habilitar análisis económico.

In [ ]:
valores = [random.randint(5000, 100000) for _ in range(114)]

In [ ]:
recoord_with_a_cost['costo'] = valores

In [ ]:
recoord_with_a_cost

### Etapa 4: Exportación
Guardamos el dataset procesado para el notebook de analytics.

In [ ]:
recoord_with_a_cost.to_csv('../data/processed/casos_recoordinados.csv', index=False)